In [4]:
import random
import time
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import CrossEncoder
from scipy.stats import pearsonr, spearmanr

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "sentence-transformers/stsb-distilroberta-base-v2"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 64 if device == "mps" else 32
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "device": device,
    "batch_size": batch_size,
    "seed": seed,
})

{'model_name': 'sentence-transformers/stsb-distilroberta-base-v2', 'dataset': 'glue/stsb', 'split': 'validation', 'device': 'mps', 'batch_size': 64, 'seed': 42}


In [5]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()
df["label"] = df["label"].astype(np.float32)

print({"num_examples": len(df), "columns": df.columns.tolist()})
print(df.head())

{'num_examples': 1500, 'columns': ['sentence1', 'sentence2', 'label']}
                              sentence1  \
0     A man with a hard hat is dancing.   
1      A young child is riding a horse.   
2  A man is feeding a mouse to a snake.   
3        A woman is playing the guitar.   
4         A woman is playing the flute.   

                                  sentence2  label  
0      A man wearing a hard hat is dancing.   5.00  
1                A child is riding a horse.   4.75  
2  The man is feeding a mouse to the snake.   5.00  
3                  A man is playing guitar.   2.40  
4                 A man is playing a flute.   2.75  


In [ ]:
model = CrossEncoder(
    model_name,
    device=device,
    max_length=256,
)
print(model_name)

config.json:   0%|          | 0.00/680 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/328M [00:00<?, ?B/s]

In [ ]:
pairs = list(zip(df["sentence1"].tolist(), df["sentence2"].tolist()))
labels = df["label"].to_numpy(dtype=np.float32)

predicted_score_0_5 = np.asarray(
    model.predict(
        pairs,
        batch_size=batch_size,
        show_progress_bar=False,
        convert_to_numpy=True,
    ),
    dtype=np.float32,
)

predicted_score_0_5 = np.clip(predicted_score_0_5, 0.0, 5.0).astype(np.float32)
absolute_error = np.abs(predicted_score_0_5 - labels).astype(np.float32)
squared_error = np.square(predicted_score_0_5 - labels).astype(np.float32)
signed_error = (predicted_score_0_5 - labels).astype(np.float32)

results_df = df.copy()
results_df["predicted_score_0_5"] = predicted_score_0_5
results_df["absolute_error"] = absolute_error
results_df["squared_error"] = squared_error
results_df["signed_error"] = signed_error
results_df["prediction_rank"] = results_df["predicted_score_0_5"].rank(method="average", ascending=True)
results_df["label_rank"] = results_df["label"].rank(method="average", ascending=True)
results_df["rank_gap"] = np.abs(results_df["prediction_rank"] - results_df["label_rank"]).astype(np.float32)

print(results_df[["sentence1", "sentence2", "label", "predicted_score_0_5", "absolute_error"]].head(10))

In [ ]:
pearson_corr = pearsonr(results_df["predicted_score_0_5"], results_df["label"]).statistic
spearman_corr = spearmanr(results_df["predicted_score_0_5"], results_df["label"]).statistic
mae = float(results_df["absolute_error"].mean())
rmse = float(np.sqrt(results_df["squared_error"].mean()))

label_bin_edges = [-0.001, 1.0, 2.0, 3.0, 4.0, 5.001]
label_bin_names = ["[0,1)", "[1,2)", "[2,3)", "[3,4)", "[4,5]"]
results_df["label_bin"] = pd.cut(
    results_df["label"],
    bins=label_bin_edges,
    labels=label_bin_names,
    include_lowest=True,
    right=False,
)

bin_agg = (
    results_df.groupby("label_bin", observed=False)
    .agg(
        count=("label", "size"),
        label_mean=("label", "mean"),
        pred_mean=("predicted_score_0_5", "mean"),
        mae=("absolute_error", "mean"),
        rmse=("squared_error", lambda x: float(np.sqrt(np.mean(x)))),
        signed_error_mean=("signed_error", "mean"),
        rank_gap_mean=("rank_gap", "mean"),
    )
    .reset_index()
)

print({
    "pearson_correlation": round(float(pearson_corr), 6),
    "spearman_correlation": round(float(spearman_corr), 6),
    "mae_0_5": round(mae, 6),
    "rmse_0_5": round(rmse, 6),
})
print(bin_agg)

In [ ]:
best_examples = (
    results_df.sort_values(
        by=["absolute_error", "rank_gap", "label"],
        ascending=[True, True, False],
    )
    [["sentence1", "sentence2", "label", "predicted_score_0_5", "absolute_error", "signed_error", "rank_gap"]]
    .head(10)
    .reset_index(drop=True)
)

worst_examples = (
    results_df.sort_values(
        by=["absolute_error", "rank_gap", "label"],
        ascending=[False, False, False],
    )
    [["sentence1", "sentence2", "label", "predicted_score_0_5", "absolute_error", "signed_error", "rank_gap"]]
    .head(10)
    .reset_index(drop=True)
)

largest_overestimates = (
    results_df.sort_values(by=["signed_error", "absolute_error"], ascending=[False, False])
    [["sentence1", "sentence2", "label", "predicted_score_0_5", "absolute_error", "signed_error"]]
    .head(10)
    .reset_index(drop=True)
)

largest_underestimates = (
    results_df.sort_values(by=["signed_error", "absolute_error"], ascending=[True, False])
    [["sentence1", "sentence2", "label", "predicted_score_0_5", "absolute_error", "signed_error"]]
    .head(10)
    .reset_index(drop=True)
)

pd.set_option("display.max_colwidth", 160)
print("BEST_AGREEMENT_EXAMPLES")
print(best_examples)
print("\nWORST_AGREEMENT_EXAMPLES")
print(worst_examples)
print("\nLARGEST_OVERESTIMATES")
print(largest_overestimates)
print("\nLARGEST_UNDERESTIMATES")
print(largest_underestimates)

In [ ]:
runtime_seconds = time.time() - start_time

summary = {
    "device_used": device,
    "model_name": model_name,
    "dataset_split": f"{dataset_name}/{dataset_config}/{split_name}",
    "num_examples": int(len(df)),
    "pearson_correlation": round(float(pearson_corr), 6),
    "spearman_correlation": round(float(spearman_corr), 6),
    "mae_0_5": round(mae, 6),
    "rmse_0_5": round(rmse, 6),
    "runtime_seconds": round(float(runtime_seconds), 2),
}

print(summary)